# Global-level clustering using functional and structural connectivity and structure-function coupling

## Purpose
This notebook performs the global-level clustering using global functional and structural connectivity and structure-function coupling features
derived from per-subject connectomes. It is intended to:
- identify data-driven subtypes within the depression cohort via hierarchical
  clustering (Ward linakge) on node-wise connectivity-strength or coupling vectors,
- quantify clustering validity and reproducibility (internal metrics +
  bootstrap stability),
- compare cluster-derived groups and the depression group to controls, and compare clusters between themselves via
  quantile regression (R / `quantreg`) with covariate adjustment,
- visualize covariate distributions and cluster assignments, and
- produce CSVs, TXT summary files and figures summarizing the analyses.

## Utility Functions (global_clustering_utils.py)
This module contains functions used to perform clustering, validation,
visualization, and statistical testing on global connectivity-derived features
for depression and control cohorts. Functions are organized broadly by
purpose: data loading, R integration, statistics, plotting, clustering,
and cross-modality agreement analyses.

## Expected Inputs
- **Per-subject connectivity data** (organized under cohort folders):
  - *Functional*: `<eid>/i2/<eid>_connectivity.npy` (2D NumPy array, correlation or z-transformed matrix).
  - *Structural*: `<eid>/i2/connectome_streamline_count_10M.csv.gz` (square CSV of streamline counts per node).
- **Cohort CSVs** (combined and depression-only):
  - `COMBINED_COHORT_PATH` – CSV with per-subject covariates (must contain at least `eid`, `depression_status`, `p21003_i2` (age), `p31` (sex)).
  - `DEPRESSION_COHORT_PATH` – CSV for the depression cohort (same expected covariates; script will attempt to merge head-motion columns if missing).
- **Head motion CSV**: `HEAD_MOTION_PATH` (may be compressed). Expected to contain `p24441_i2` (fMRI) and/or `p24453_i2` (dMRI) per `eid`.
- **Optional**: pre-computed combined cluster CSVs for cross-modality agreement.

## High-level Steps
1. **Setup & I/O**: Read configuration constants and prepare output directories.
2. **Load cohorts**: Read combined and depression cohort CSVs; merge head-motion if needed.
3. **Compute connectivity**: 
   - Load matrices and compute per-node "strength" (weighted degree).
   - For SFC, compute correlation between FC and SC profiles.
   - *Note*: NaNs are tracked and treated as zeros for clustering.
4. **Pre-clustering diagnostics** (Optional): Check specific feature-covariate associations.
5. **Hierarchical clustering**: Ward linkage on depression cohort features (k=2).
6. **Validation**: Silhouette/Calinski-Harabasz scores and Bootstrap stability analysis.
7. **Cross-modality agreement**: Compare clusters across functional, structural, and SFC modalities.
8. **Covariate plotting**: Summarize Age, Sex, Motion, and Comorbidities by Group/Cluster.
9. **Quantile Regression (R)**: Compare groups (Depression vs Control, Clusters vs Control) using median regression with covariate adjustment.
10. **Visualization**: Generate violin plots with FDR-corrected significance bars.

## Outputs Produced 
- **CSVs**: Cluster assignments, bootstrap diagnostics, stability scores, pairwise agreement.
- **TXT**: Summary table of covariate distributions and test results.
- **Figures (PNG)**: Dendrograms, stability plots, covariate panels, violin plots, agreement heatmaps.
- **Logs**: Console output and FDR correction tables are saved text files.

## Requirements
- Python packages: `numpy`, `pandas`, `matplotlib`, `seaborn`, `scipy`, `sklearn`, `statsmodels`, `rpy2`
- R (system installation) with packages: `quantreg`, `multcomp`

## Side Effects
- Writes output files (CSV, PNG, TXT) to provided output directories.
- `run_quantile_regression` writes a temporary CSV and invokes R.

In [ ]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================

import os

# Data paths (Adjust these paths as needed)
DEPRESSION_DIR = '.../F32_notask_STRCO_RSSCHA_RSTIA'
CONTROL_DIR = '.../control_notask_STRCO_RSSCHA_RSTIA'
GENERAL_DIR = '.../cohorts'
COMBINED_COHORT_PATH = '.../combined_cohort_F32.csv'
DEPRESSION_COHORT_PATH = '.../depression_cohort_F32.csv'
HEAD_MOTION_PATH = '.../head_motion.csv.gz'

# Output directories
VALIDATION_PLOTS_BASE_DIR = '.../schaefer1000+tian54'
FIGURES_BASE_DIR = '.../schaefer1000+tian54'
COHORTS_DIR = '.../cohorts'

# Analysis Parameters
CONNECTIVITY_TYPES = ['functional', 'structural', 'sfc']
ICD_COVARIATES = ['I10', 'Z864', 'F419']
PROGRESS_EVERY = 50
# Bootstrap iterations for quantile regression (R) standard errors and confidence intervals
QUANTILE_REGRESSION_BOOTSTRAP_R = 5000
# Bootstrap iterations for clustering stability analysis
CLUSTER_STABILITY_BOOTSTRAP_ITER = 5000
DEPENDENT_VAR = 'Connectivity'
CLUSTER_COL = 'Cluster'
GROUP_COL = 'Group'

# Motion metrics columns
fMRI_MOTION_METRIC = 'p24441_i2'
dMRI_MOTION_METRIC = 'p24453_i2'

In [ ]:
# ==============================================================================
# IMPORTS
# ==============================================================================

import sys
import io
from contextlib import contextmanager
import pandas as pd
import numpy as np

# Add source_code to path to allow importing modules
sys.path.append(os.path.abspath('../source_code'))

# Import utility functions from global_clustering_utils
from clusters.global_clustering_utils import (
    setup_r_environment,
    load_single_connectivity_matrix,
    compute_node_strength_from_matrix,
    compute_sfc_features_from_matrices,
    plot_dendrogram,
    perform_hierarchical_clustering,
    plot_bootstrap_diagnostics,
    ensure_2d_array,
    subject_scalar_summary,
    get_motion_columns,
    load_and_prepare_cohort_data,
    perform_clustering_validation,
    save_clustering_results,
    analyze_cross_modality_agreement,
    create_combined_dataframe,
    merge_combined_cohort_connectivity_clusters,
    determine_covariate_distributions,
    run_quantile_regression,
    apply_multiple_testing_correction,
    create_violin_plot_with_significance,
)

# Setup Output Capture
class Tee(io.TextIOBase):
    """File-like object that writes to both stdout and buffer."""
    def __init__(self, original, buffer):
        self.original = original
        self.buffer = buffer
        
    def write(self, s):
        self.original.write(s)
        self.buffer.write(s)
        return len(s)
        
    def flush(self):
        self.original.flush()
        self.buffer.flush()

@contextmanager
def capture_stdout_to_log(log_path):
    """Mirror stdout to a buffer and persist it to disk when the block ends."""
    original_stdout = sys.stdout
    buffer = io.StringIO()
    sys.stdout = Tee(original_stdout, buffer)
    try:
        yield buffer
    finally:
        sys.stdout = original_stdout
        with open(log_path, 'w') as log_file:
            log_file.write(buffer.getvalue())

## Step 1: Initialize R Environment
The analysis requires R for quantile regression (`quantreg` package). This step initializes the `rpy2` interface and checks for necessary R packages.

In [ ]:
print("[STEP 1/9] Initializing R environment...")
setup_r_environment()
print("R packages loaded: quantreg, base, utils, stats")

## Step 2: Load Cohort Data
Load the combined and depression cohort CSVs. If head motion data is missing from the cohort files, it will be merged from the head motion file.

In [ ]:
print("\n[STEP 2/9] Loading cohort data...")
cohort_data = load_and_prepare_cohort_data(
    COMBINED_COHORT_PATH, DEPRESSION_COHORT_PATH, HEAD_MOTION_PATH, save_if_modified=True
)
print(f"  Depression subjects: {len(cohort_data['depression_subject_ids'])}")
print(f"  Control subjects: {len(cohort_data['control_subject_ids'])}")

## Analysis Pipeline Function
This function encapsulates the steps for a single connectivity type (e.g., 'functional', 'structural', 'sfc'). It corresponds to Steps 3-9 in the main script.

In [ ]:
def execute_pipeline_for_conn_type(
    conn_type, cohort_data, validation_plots_dir, figures_base_dir, 
    motion_columns, primary_motion_metric, fdr_log_path, 
    icd_covariates, dependent_var, cluster_col, group_col
):
    """Run the full pipeline for a specific connectivity modality."""

    combined_data = cohort_data['combined_data']
    depression_subject_ids = list(cohort_data['depression_subject_ids'])
    control_subject_ids = list(cohort_data['control_subject_ids'])

    # ------------------------------------------------------------------
    # STEP 3: Load connectivity data
    # ------------------------------------------------------------------
    print(f"\n[STEP 3/9] Computing {conn_type} connectivity measure data...")
    
    # ------------------------------------------------------------------
    # STEP 3a: Functional / Structural
    # ------------------------------------------------------------------
    # For 'functional' and 'structural' modalities we load single connectivity
    # matrices per subject and compute a normalized node-wise strength vector.
    if conn_type in ('functional', 'structural'):
        dep_features = [] # list of per-subject normalized node-strength vectors (depression)
        dep_ids =  [] # corresponding subject IDs (keeps only subjects with data)
        total_dep = len(depression_subject_ids)
        for i, sid in enumerate(depression_subject_ids, start=1):
            # Attempt to load the subject's connectivity matrix; may return None if missing
            mat = load_single_connectivity_matrix(sid, DEPRESSION_DIR, conn_type)
            if mat is None: 
                # Skip subjects without a valid matrix
                continue
            # Compute normalized node strength (global feature) from the connectivity matrix
            dep_features.append(compute_node_strength_from_matrix(mat, conn_type=conn_type))
            dep_ids.append(sid)
            # Periodic progress reporting to keep the user informed on long runs
            if PROGRESS_EVERY > 0 and (i % PROGRESS_EVERY == 0 or i == total_dep):
                print(f"  Progress: {i}/{total_dep} {conn_type} depression subjects")

        # Repeat for control cohort
        ctrl_features = [] # list of per-subject normalized node-strength vectors (control)
        ctrl_ids = [] # corresponding subject IDs (keeps only subjects with data)
        total_ctrl = len(control_subject_ids)
        for i, sid in enumerate(control_subject_ids, start=1):
            mat = load_single_connectivity_matrix(sid, CONTROL_DIR, conn_type)
            if mat is None: 
                # Skip subjects without a valid matrix
                continue
            ctrl_features.append(compute_node_strength_from_matrix(mat, conn_type=conn_type))
            ctrl_ids.append(sid)
            if PROGRESS_EVERY > 0 and (i % PROGRESS_EVERY == 0 or i == total_ctrl):
                print(f"  Progress: {i}/{total_ctrl} {conn_type} control subjects")

        # If either group has no matrices, there's nothing to cluster for this modality
        if not dep_features or not ctrl_features:
            print("  No connectivity matrices found for this modality; skipping.")
            return
        
        # Replace the subject ID lists with the filtered lists (subjects with data)
        depression_subject_ids = dep_ids
        control_subject_ids = ctrl_ids
        
        # Basic sanity check: all normalized node-strength vectors should have identical length
        n_nodes = dep_features[0].shape[0]
        if any(vec.shape[0] != n_nodes for vec in ctrl_features):
            raise ValueError(f"Node count mismatch dep={n_nodes} ctrl={ctrl_features[0].shape[0]}")
        
        print(f"  Loaded matrices for {len(dep_features)} depressed and {len(ctrl_features)} control subjects")
        
        # Expose for downstream steps: hierarchical clustering & summaries
        global_features_depression = dep_features
        global_features_control = ctrl_features

    # ------------------------------------------------------------------
    # STEP 3b: Structure-Function Coupling (SFC)
    # ------------------------------------------------------------------
    # For SFC we need both functional and structural matrices for the same subject.
    # We compute a coupling vector from paired FC/SC matrices.
    else:
        dep_features, dep_ids = [], []
        sfc_nan_info_dep = pd.DataFrame({"eid": [], "percent_nan": []}) # To track % of NaNs in SFC vectors per depression subject
        total_dep = len(depression_subject_ids)
        
        for i, sid in enumerate(depression_subject_ids, start=1):
            # Load both FC and SC; skip subject if either modality is missing
            fc_mat = load_single_connectivity_matrix(sid, DEPRESSION_DIR, 'functional')
            sc_mat = load_single_connectivity_matrix(sid, DEPRESSION_DIR, 'structural')
            if fc_mat is None or sc_mat is None: 
                continue
            # compute_sfc_features_from_matrices expects lists and returns
            # a list of per-subject vectors plus percent_nan. Unpack the
            # returned list and append the first (and only) vector so that
            # downstream code receives a 1D NumPy array (with .shape).
            sfc_list, percent_nan = compute_sfc_features_from_matrices([fc_mat], [sc_mat])
            sfc_vec = sfc_list[0]
            sfc_nan_info_dep = pd.concat([sfc_nan_info_dep, pd.DataFrame({"eid": [sid], "percent_nan": [percent_nan]})])
            dep_features.append(sfc_vec)
            dep_ids.append(sid)
            if PROGRESS_EVERY > 0 and (i % PROGRESS_EVERY == 0 or i == total_dep):
                print(f"  Progress: {i}/{total_dep} SFC depression subjects")

        ctrl_features, ctrl_ids = [], []
        sfc_nan_info_ctrl = pd.DataFrame({"eid": [], "percent_nan": []}) # To track % of NaNs in SFC vectors per control subject
        total_ctrl = len(control_subject_ids)
        
        for i, sid in enumerate(control_subject_ids, start=1):
            fc_mat = load_single_connectivity_matrix(sid, CONTROL_DIR, 'functional')
            sc_mat = load_single_connectivity_matrix(sid, CONTROL_DIR, 'structural')
            if fc_mat is None or sc_mat is None: continue
            
            sfc_list, percent_nan = compute_sfc_features_from_matrices([fc_mat], [sc_mat])
            sfc_vec = sfc_list[0]
            sfc_nan_info_ctrl = pd.concat([sfc_nan_info_ctrl, pd.DataFrame({"eid": [sid], "percent_nan": [percent_nan]})])
            ctrl_features.append(sfc_vec)
            ctrl_ids.append(sid)
            if PROGRESS_EVERY > 0 and (i % PROGRESS_EVERY == 0 or i == total_ctrl):
                print(f"  Progress: {i}/{total_ctrl} SFC control subjects")

        # Concatenate and save NaN info for transparency; this can help identify if certain subjects have unreliable SFC estimates due to many NaN values (e.g., from constant FC/SC profiles)
        sfc_nan_info = pd.concat([sfc_nan_info_dep, sfc_nan_info_ctrl], ignore_index=True)
        sfc_nan_info.to_csv(os.path.join(GENERAL_DIR, 'sfc_vector_nan_info.csv'), index=False)

        # Need overlapping FC+SC subjects in both groups to compute SFC clusters
        if not dep_features or not ctrl_features:
            print("  Missing overlapping FC/SC subjects; skipping SFC.")
            return

        # Update subject ID lists to the subset with both modalities present
        depression_subject_ids = dep_ids
        control_subject_ids = ctrl_ids

        # Ensure vector lengths match across groups
        n_nodes = dep_features[0].shape[0]
        if any(vec.shape[0] != n_nodes for vec in ctrl_features):
            raise ValueError("FC/SC node count mismatch for Structure-Function Coupling")

        print(f"  Loaded FC+SC matrices for {len(dep_features)} depressed and {len(ctrl_features)} control subjects")

        # Expose for downstream steps: hierarchical clustering & summaries
        global_features_depression = dep_features
        global_features_control = ctrl_features

    # ------------------------------------------------------------------
    # STEP 4: Hierarchical clustering
    # ------------------------------------------------------------------
    print("\n[STEP 4/9] Hierarchical clustering (Ward linkage, k=2)...")
    X_mat = ensure_2d_array(np.asarray(global_features_depression))
    Z, clusters = perform_hierarchical_clustering(X_mat, n_clusters=2)

    print("  Cluster sizes:")
    print(f"    Cluster 0: {np.sum(clusters == 0)}")
    print(f"    Cluster 1: {np.sum(clusters == 1)}")

    plot_dendrogram(Z, conn_type, figures_base_dir)

    subject_scalar_depression = subject_scalar_summary(global_features_depression)
    subject_scalar_control = subject_scalar_summary(global_features_control)

    # ------------------------------------------------------------------
    # STEP 5: Clustering validation
    # ------------------------------------------------------------------
    print("\n[STEP 5/9] Clustering validation...")
    validation_results = perform_clustering_validation(
        X_mat, Z, clusters, conn_type, validation_plots_dir, CLUSTER_STABILITY_BOOTSTRAP_ITER
    )
    save_clustering_results(
        validation_results['stability_results'], clusters, subject_scalar_depression,
        conn_type, validation_plots_dir
    )
    plot_bootstrap_diagnostics(
        validation_results['stability_results'], clusters, conn_type,
        validation_plots_dir, analysis_level='global'
    )

    # ------------------------------------------------------------------
    # STEP 6: Cross-modality agreement (Runs for all types, but depends on files existing)
    # ------------------------------------------------------------------
    print("\n[STEP 6/9] Cross-connectivity-type cluster agreement...")
    available_types = analyze_cross_modality_agreement(COHORTS_DIR, validation_plots_dir)

    # ------------------------------------------------------------------
    # STEP 7: Covariate distribution visualization
    # ------------------------------------------------------------------
    print("\n[STEP 7/9] Covariate distribution visualization...")
    combined_df = create_combined_dataframe(
        control_subject_ids, depression_subject_ids,
        subject_scalar_control, subject_scalar_depression,
        clusters, combined_data, conn_type=conn_type
    )

    determine_covariate_distributions(
        combined_df, available_types, conn_type, primary_motion_metric,
        validation_plots_dir, COHORTS_DIR, icd_covariates,
    )

    combined_df.to_csv(
        os.path.join(COHORTS_DIR, f'combined_cohort_F32_global_{conn_type}_connectivity_clusters.csv'),
        index=False
    )

    # ------------------------------------------------------------------
    # STEP 8: Quantile regression analysis (R)
    # ------------------------------------------------------------------
    print("\n[STEP 8/9] Quantile regression analysis...")
    motion_covariate_list = list(motion_columns.values())
    results = run_quantile_regression(
        combined_df, conn_type, icd_covariates,
        motion_covariates=motion_covariate_list,
        tau=0.5, R=QUANTILE_REGRESSION_BOOTSTRAP_R,
        dependent_var=dependent_var, cluster_col=cluster_col, group_col=group_col,
    )

    _, corrected_p_values = apply_multiple_testing_correction(
        p_values=list(results['p_values'].values()),
        test_methods=[
            'QR (median) depression vs control',
            'QR (median) cluster 0 vs control',
            'QR (median) cluster 1 vs control',
            'QR (median) cluster 0 vs cluster 1'
        ],
        variable_names=[f'Global {conn_type.capitalize()} Connectivity'] * 4,
        log_path=fdr_log_path
    )

    # ------------------------------------------------------------------
    # STEP 9: Violin plot with significance brackets
    # ------------------------------------------------------------------
    print("\n[STEP 9/9] Creating violin plot with significance brackets...")
    create_violin_plot_with_significance(
        subject_scalar_control, subject_scalar_depression, clusters,
        corrected_p_values, conn_type,
        os.path.join(validation_plots_dir, 'individual_avg_conn_clusters_violinplot.png')
    )

    print(f"\nCompleted {conn_type} connectivity analysis")

## Execute Pipeline Modality-by-Modality
Iterate through each configured connectivity type ('functional', 'structural', 'sfc') and run the pipeline. 

This loop mirrors the main script execution. Logs are captured to text files.

In [ ]:
for conn_type in CONNECTIVITY_TYPES:
    # Prepare per-modality directories
    validation_plots_dir = os.path.join(VALIDATION_PLOTS_BASE_DIR, f'{conn_type}_con')
    figures_dir = os.path.join(FIGURES_BASE_DIR, f'{conn_type}_con')
    os.makedirs(validation_plots_dir, exist_ok=True)
    os.makedirs(figures_dir, exist_ok=True)

    # Prepare logs and metrics
    general_log_path = os.path.join(DEPRESSION_DIR, f'global_{conn_type}_connectivity_output.txt')
    fdr_log_path = os.path.join(DEPRESSION_DIR, f'global_{conn_type}_connectivity_FDR.txt')
    
    motion_columns = get_motion_columns(conn_type, fMRI_MOTION_METRIC, dMRI_MOTION_METRIC)
    primary_motion_metric = next(iter(motion_columns.values()))

    display_conn = "Structure-Function Coupling" if str(conn_type).lower() == "sfc" else str(conn_type).upper()
    print(f"\n{'='*80}")
    print(f"Global-level Depression Subtyping Pipeline — {display_conn}")
    print(f"{'='*80}\n")

    # Run pipeline with output capturing
    with capture_stdout_to_log(general_log_path):
        execute_pipeline_for_conn_type(
            conn_type=conn_type,
            cohort_data=cohort_data,
            validation_plots_dir=validation_plots_dir,
            figures_base_dir=FIGURES_BASE_DIR,
            motion_columns=motion_columns,
            primary_motion_metric=primary_motion_metric,
            fdr_log_path=fdr_log_path,
            icd_covariates=ICD_COVARIATES,
            dependent_var=DEPENDENT_VAR,
            cluster_col=CLUSTER_COL,
            group_col=GROUP_COL,
        )
    
    print(f"Saved run log to: {general_log_path}")
    print(f"Saved FDR correction log to: {fdr_log_path}")

## Merge Cluster Results
After running for all available modalities, merge the cluster assignment results into a single CSV.

In [ ]:
merged_df = merge_combined_cohort_connectivity_clusters(COHORTS_DIR, conn_types=CONNECTIVITY_TYPES)
merged_path = os.path.join(COHORTS_DIR, 'global_merged_connectivity_clusters.csv')
merged_df.to_csv(merged_path, index=False)
print(f"\n{'='*80}")
print(f"Saved merged connectivity clusters to: {merged_path}")
print("ALL MODALITIES COMPLETE")